# Stage 4 — pdf_to_text

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and a verification step. The implementation itself is yours to write.


## 1. Setup

Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running Stage 0 first in the same runtime), run all of them now.


### 1a. Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### 1b. Install dependencies

Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### 1c. (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


### 1d. Stage the bundle-shipped configs


In [ ]:
# Copy bundle-shipped configs into the directories each stage script expects.
# Each stage's input.txt / criteria.txt / schema.json lives under configs/
# in the repo; the actual scripts read them relative to cwd.
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
shutil.copy("configs/schema.json",           "schema.json")
print("configs staged")


### 1e. Load prior stages' reference outputs

Stage 4 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_03/data/` from the canonical reference run so Stage 4 has inputs to work with.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 4.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <4 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 4):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini

Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Convert every artifact under `stage_03/data/` to a `.md` under
`stage_04/data/`. Handle three input shapes uniformly:

  *.pdf  → run `lit parse <path> -o <tmp>.txt` and read the result
  *.xml  → walk JATS: emit `# <title>`, `## Abstract`, paragraphs, and
           heading-rendered <sec> structure. Drop figures, refs, math.
  *.json → render Stage 3 metadata fallback as title + abstract +
           authors/journal footer.

Both XML and metadata-fallback outputs should emit an authors/journal/
year/pub_types footer so the Stage 5 extractor sees a consistent shape.
```


## 3. Gotchas Gemini probably won't know

Copy any that apply into Gemini if it goes off-track:

- **liteparse is a Node CLI, not a Python lib.** Invoke via
  `subprocess.run(["lit", "parse", path, "-o", tmp_path], check=True)`.
  Was installed in Stage 0.
- **JATS uses `<sec>` and `<p>`.** Recurse with `element.iter()` or a
  depth-tracking walker — don't grab only top-level paragraphs.
- **Authors live in `<front>//<contrib-group>//<contrib>`** in JATS,
  with `<surname>` + `<given-names>` children. Bodies have no author
  byline — without a footer, Stage 5 hallucinates first_author.
- **`itertext()` again.** Same trick as Stage 1 for any element that
  might wrap inline children.
- **Floor each output.** Fulltext output should be much larger than an
  abstract — assert `getsize > 5_000` (or `> 500` for the JSON
  fallback) to catch parse failures early.


## 4. Seed — a few lines to anchor Gemini in the right direction


In [ ]:
import glob, json, os, re, subprocess, tempfile
import xml.etree.ElementTree as ET

STAGE = "stage_04"
DATA = f"{STAGE}/data"
IN_DATA = "stage_03/data"
os.makedirs(DATA, exist_ok=True)


## 5. Your implementation

Drive Gemini to fill this in. Iterate until the verification cell below passes.


In [ ]:
# TODO: implement Stage 4 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the verification cell next.


## 6. Verify


In [ ]:
import glob, os
mds = sorted(glob.glob("stage_04/data/*.md"))
assert mds, "no .md files produced"
for p in mds:
    assert os.path.getsize(p) > 500, f"{p}: suspiciously small"
print(f"OK — {len(mds)} .md files")


## 7. Run the eval grader

The eval reads only your stage's output and writes `stage_04/eval/eval_*.json` + `score.json`.


In [ ]:
!python eval/eval_04_script.py


## 8. Stuck? Skip this stage

Copy the reference run's Stage 4 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil, glob
os.makedirs("stage_04/data", exist_ok=True)
for src in glob.glob("reference_outputs/stage_04/data/*.md"):
    shutil.copy(src, "stage_04/data/")
print(f"copied {len(os.listdir('stage_04/data'))} reference .md files")
